<div align="center">

# Prepare weighted sentence vectors

Make classic word vectors more useful for sentence search.

</div>

## When do you need this?

You already have GloVe word vectors and want better sentence comparisons.
You are moving a SIF or uSIF model from fse 1.x to 2.0.
**Prepare the model once**, then use fse to run the finished table.

## The story in pictures

![Training texts teach word weights and common directions. Fold these into the word table, then encode new sentences.](media/01-sif-usif.svg)

## What changes?

A plain mean gives every word the same weight.
**SIF** (Smooth Inverse Frequency) lowers the weight of common words and removes shared directions from sentence vectors.
**uSIF** estimates the word-weight setting from your corpus and scales the directions it removes.
A direction here is a pattern of numbers shared by many sentences.

## 1. Read sentence pairs

[STS Benchmark](https://ixa2.si.ehu.eus/stswiki/index.php/STSbenchmark) contains sentence pairs with human similarity scores.
We use training texts to prepare the model and keep **test scores out of preparation**.
The shared helper handles temporary folders and downloads, as described in the [index](README.md#run-a-recipe).

In [1]:
from _helpers import Recipe
import fse
import numpy as np

recipe = Recipe()
work = recipe.work

Recipe run: 2026-11-01. Generated files are temporary.
Apple M1 Pro. macOS-27.0.1-arm64-arm-64bit-Mach-O. Python 3.13.16.


In [2]:
from _sif import read_pairs, load_tables, prepare, means, remove, unit, pearson_score
from IPython.display import Markdown, display

train_a, train_b, _ = read_pairs("train")
test_a, test_b, gold = read_pairs("test")
tables = load_tables(recipe)
print({"train_pairs": len(train_a), "test_pairs": len(test_a)})

/Users/oliverborchers/.herdr/worktrees/Fast_Sentence_Embeddings/dr195-tutorials/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'train_pairs': 5749, 'test_pairs': 1379}


## 2. Weight words and remove common patterns

The helper implements the [SIF paper](https://openreview.net/forum?id=SyK00v5xx), [uSIF paper](https://aclanthology.org/W18-3012/), and [fse 1.x choices](https://github.com/oborchers/Fast_Sentence_Embeddings/tree/1.x/src/fse/models).
It finds common directions without first subtracting the average vector, matching those choices.
**Folding** means applying the learned changes to every word row ahead of time.
After folding, averaging those rows gives the changed sentence vector, apart from rounding.

## 3. Save, reload, and compare

We try a plain mean, SIF with 10 directions, and uSIF with 5 directions on both tables.
GloVe splits lowercase words with a regular expression, while POTION keeps its own word-piece tokenizer.
**Keep that tokenizer with the saved model** because these artifacts expect token lists, rather than raw strings.
The loop also checks folded output against changes applied after averaging.

In [3]:
results = []
for name, table, words, tokenize in tables:
    train_tokens = [tokenize(t) for t in train_a + train_b]
    test_tokens = [tokenize(t) for t in test_a + test_b]
    for method in ("mean", "sif", "usif"):
        folded, weighted, basis, factors = prepare(table, words, train_tokens, method)
        finished = fse.Encoder.from_arrays(folded, words=words, tokenizer="external", normalize=True)
        path = work / f"{name}-{method}.safetensors"
        finished.save(path)
        finished = fse.Encoder.load(path, mmap=True)
        encoded = finished.encode(test_tokens, workers=1)
        reference = unit(remove(means(weighted, words, test_tokens), basis, factors))
        error = float(np.max(np.abs(encoded - reference)))
        np.testing.assert_allclose(encoded, reference, atol=3e-5, rtol=3e-5)
        results.append({"table": name, "method": method,
                        "pearson_x100": pearson_score(encoded, gold), "fold_max_abs_error": error})
        del finished, weighted, folded

The table shows quality for each finished model on the same unseen sentence pairs.
**The error check tests folding**, rather than assuming the saved model is equivalent.

In [4]:
display(Markdown("| Table | Method | Pearson × 100 |\n|---|---|---:|\n" + "\n".join(
    f"| {r['table']} | {r['method']} | {r['pearson_x100']:.2f} |" for r in results
)))
print("Largest folding error:", max(r["fold_max_abs_error"] for r in results))

| Table | Method | Pearson × 100 |
|---|---|---:|
| GloVe-300 | mean | 45.09 |
| GloVe-300 | sif | 70.67 |
| GloVe-300 | usif | 66.28 |
| POTION-8M | mean | 77.26 |
| POTION-8M | sif | 75.71 |
| POTION-8M | usif | 76.91 |

Largest folding error: 1.0356307029724121e-06


## Result

Pearson correlation measures how closely the predicted scores follow human scores, and higher is better here.
**Compare methods within each table** because the tokenizers differ between GloVe and POTION.
POTION already contains learned weights, so extra SIF or uSIF is an experiment, not a default recommendation.
These results concern this split only, so evaluate your own texts before changing a deployed model.

## Optional: check the old README numbers

The old experiment prepared its directions using test texts and English word frequencies.
That makes it a **historical diagnostic**, rather than a test of unseen data.
The next cell keeps it separate from the training-only comparison above.

In [5]:
from _sif import historical_check

_, glove_table, glove_words, _ = tables[0]
legacy_results = historical_check(recipe, glove_table, glove_words, test_a, test_b, gold)
for result in legacy_results:
    print(result)

{'method': 'mean', 'recipe_pearson_x100': 44.46410145406453, 'README_1_1_0': 44.46, 'delta_points': 0.004101454064532106}
{'method': 'sif', 'recipe_pearson_x100': 71.3791648937449, 'README_1_1_0': 71.43, 'delta_points': -0.05083510625510712}
{'method': 'usif', 'recipe_pearson_x100': 68.81903532181575, 'README_1_1_0': 68.82, 'delta_points': -0.0009646781842462815}


The historical targets are mean 44.46, SIF 71.43, and uSIF 68.82.
The printed differences show whether this run reproduces each target.
**A small numerical difference is still a difference**, so do not claim exact reproduction when the numbers disagree.

## What next?

- [Run a trained linear head](02-linear-heads.ipynb).
- [fse API and design](../docs/architecture/formats.md)
- [GloVe source model and license](https://huggingface.co/fse/glove-wiki-gigaword-300), [POTION model and license](https://huggingface.co/minishlab/potion-base-8M)

Back to the [recipe index](README.md) or the [main README](../README.md).

## Clean up

The helper keeps generated files in a temporary folder. Run this after you finish exploring the results.

In [6]:
recipe.close()